## adsorption energy screening

(Cu, Ag, Au, Pt), data from Materials Project

In [51]:
from pymatgen.core.surface import SlabGenerator
from pymatgen.io.ase import AseAtomsAdaptor
from ase.calculators.emt import EMT
from mace.calculators import mace_mp
from pymatgen.analysis.adsorption import AdsorbateSiteFinder

In [40]:
# query Materials Project API for DFT data

import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("MATERIALS_PROJECT_API_KEY")

from mp_api.client import MPRester

In [41]:
# materials ids: Cu = mp-30, Ag = mp-124, Au = mp-81, Pt = mp-126
# (all cubic, from space group Fm3m)

with MPRester(api_key) as mpr:
    docs = mpr.summary.search(material_ids=["mp-30", "mp-124", "mp-81", "mp-126"])

/var/folders/nq/bjj6kssx74500m34s24x3mf00000gn/T/ipykernel_26219/608630817.py:5: DeprecationWarning: Accessing summary data through MPRester.summary is deprecated. Please use MPRester.materials.summary instead.
  docs = mpr.summary.search(material_ids=["mp-30", "mp-124", "mp-81", "mp-126"])


Retrieving SummaryDoc documents:   0%|          | 0/4 [00:00<?, ?it/s]

In [12]:
# inspect column names

len(docs)  # check materials list length

from pprint import pprint

for k in docs[0].model_dump().keys():
    print(k)

has_props
shape_factor
dos_energy_up
energy_per_atom
e_total
formula_pretty
equilibrium_reaction_energy_per_atom
e_electronic
nsites
material_id
num_unique_magnetic_sites
total_magnetization_normalized_formula_units
shear_modulus
is_stable
theoretical
band_gap
efermi
dos
homogeneous_poisson
ordering
energy_above_hull
uncorrected_energy_per_atom
grain_boundaries
weighted_surface_energy_EV_PER_ANG2
num_magnetic_sites
chemsys
formation_energy_per_atom
surface_anisotropy
volume
deprecated
vbm
e_ij_max
origins
is_gap_direct
last_updated
deprecation_reasons
density
composition
weighted_surface_energy
structure
decomposes_to
is_metal
e_ionic
task_ids
total_magnetization_normalized_vol
bulk_modulus
database_IDs
warnings
bandstructure
property_name
formula_anonymous
cbm
xas
total_magnetization
universal_anisotropy
density_atomic
nelements
n
elements
es_source_calc_id
possible_species
symmetry
dos_energy_down
has_reconstructed
builder_meta
is_magnetic
types_of_magnetic_species
weighted_work_func

In [ ]:
# # view this is VESTA
# from ase.io import write

# write('ag-super.cif', supers[1])

In [52]:
# convert pymatgen structures from MP data to ase slabs objects
# build slabs w/ pymatgen to make geometrically correct structure
# find adsorption sites w/ pymatgen

N = 4
slabs = [None] * N
site_data = [None] * N

for n in range(N):
    init_struct = docs[n].structure

    sg = SlabGenerator(
        initial_structure=init_struct,
        miller_index=(1, 1, 1),
        min_slab_size=10,
        min_vacuum_size=10
    )

    slab = sg.get_slabs()[0]
    asf = AdsorbateSiteFinder(slab)  # create adsorption site finder
    sites = asf.find_adsorption_sites()  # find sites
    site_data[n] = sites
    
    slabs[n] = AseAtomsAdaptor.get_atoms(slab)  # convert slabs to ASE


/Users/zschwab/miniconda3/envs/mlip/lib/python3.11/site-packages/pymatgen/core/surface.py:1022: FutureWarning: lcm is deprecated, and will be removed on 2028-01-01
Use math.lcm instead.
  lcm_miller = lcm(*(miller_index[i] for i, _d in non_orth_ind))
/Users/zschwab/miniconda3/envs/mlip/lib/python3.11/site-packages/pymatgen/core/surface.py:1022: FutureWarning: lcm is deprecated, and will be removed on 2028-01-01
Use math.lcm instead.
  lcm_miller = lcm(*(miller_index[i] for i, _d in non_orth_ind))
/Users/zschwab/miniconda3/envs/mlip/lib/python3.11/site-packages/pymatgen/core/surface.py:1022: FutureWarning: lcm is deprecated, and will be removed on 2028-01-01
Use math.lcm instead.
  lcm_miller = lcm(*(miller_index[i] for i, _d in non_orth_ind))
/Users/zschwab/miniconda3/envs/mlip/lib/python3.11/site-packages/pymatgen/core/surface.py:1022: FutureWarning: lcm is deprecated, and will be removed on 2028-01-01
Use math.lcm instead.
  lcm_miller = lcm(*(miller_index[i] for i, _d in non_orth_in

In [53]:
# define calculators (EMT, MACE)

emt_calc = EMT()
mace_calc = mace_mp(model="medium", dispersion=True, default_dtype="float32")

/Users/zschwab/miniconda3/envs/mlip/lib/python3.11/site-packages/mace/calculators/mace.py:226: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Using Materials Project MACE for MACECalculator with /Users/zschwab/.cache/mace/20231203mace128L1_epoch199model
Using float32 for MACECalculator, which is faster but less accurate. Recommended for MD. Use float64 for geometry optimization.
Using TorchDFTD3Calculator for D3 dispersion corrections


In [54]:
# evaluation function

def evaluate(atoms, calc):
    atoms = atoms.copy()
    atoms.calc = calc

    return {
        "energy": atoms.get_potential_energy(),
        "forces": atoms.get_forces(),
        "stress": atoms.get_stress(),
    }

In [ ]:
# inspect sites
print(site_data[0].keys())
print(site_data[0]["hollow"])  # gives coord of first hollow site
# gives 2 coords b/c there's fcc and hcp hollows

dict_keys(['ontop', 'bridge', 'hollow', 'all'])
[array([ 2.52962531,  0.03777679, 13.85951394]), array([ 1.26481265, -0.55846174, 13.43790979])]


In [ ]:
# add adsorbate (CO) to a hollow site on each slab
from ase.build import molecule, add_adsorbate

ads_slabs = [None] * N

for n in range(N):
    ads_slabs[n] = slabs[n].copy()
    
    hollow_site = site_data[n]["ontop"][0]  # select first top site
    add_adsorbate(
        ads_slabs[n], 
        molecule('CO'),
        height=2.0,
        position=hollow_site[:2]  # pymatgen gives [x,y,z] but ASE expects (x,y)
    )

In [ ]:
# run eval

emt_slab_eval = [None] * N
mace_slab_eval = [None] * N
emt_ads_eval = [None] * N
mace_ads_eval = [None] * N

for n in range(N):
    emt_slab_eval[n] = evaluate(slabs[n], emt_calc)
    mace_slab_eval[n] = evaluate(slabs[n], mace_calc)
    emt_ads_eval[n] = evaluate(ads_slabs[n], emt_calc)
    mace_ads_eval[n] = evaluate(ads_slabs[n], mace_calc)